# Risk/UQ Paper Track: Risk Model Training

## Objective
Train and calibrate the CRAR risk ensemble with full Colab resumability and persistent artifacts.

## Expected Outputs
- Risk dataset and summary artifacts
- Ensemble checkpoints and model metadata
- Calibration outputs (temperature + conformal)
- Reliability diagnostics and validation predictions
- Notebook contract manifest with gate and completion states


## Hypotheses Being Tested
- **H1:** Post-hoc temperature scaling improves reliability (ECE/NLL) vs raw probabilities.
- **H2:** Calibrated `failure_proxy_h15` remains usable under configured shift suites.
- **H3:** Resume protocol prevents progress loss under Colab runtime interrupts.


## Methodology (Contract-Aligned)
1. Deterministic bootstrap + Drive mount + repo sync.
2. Single-source config and run context setup.
3. Mandatory quick probe gate.
4. Mandatory preflight gate.
5. Resume-aware dataset + training + calibration execution.
6. Diagnostics and manifest updates for reproducibility.


## Step 1 - Bootstrap Environment (Deterministic Setup + Drive + Repo)
Use the same runtime bootstrap pattern as the closed-loop simulation notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Required Config Block + Run Context
This block is the single source of truth for run identity and resume behavior.


In [ ]:
from src.workflows import initialize_run_context, report_run_context

RUN_TAG = ''
RUN_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'

N_SHARDS = 1
SHARD_ID = 'auto'

AUTO_RUN_MAIN_LOOP_WHEN_READY = False
RUN_MAIN_LOOP_OVERRIDE = False

RUN_MODE = 'auto'  # auto | fresh | resume
WARN_ON_CONFIG_DRIFT = True

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
RUN_TAG = run_context.run_tag
SHARD_ID = run_context.shard_id
run_prefix = run_context.run_prefix

print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)


## Step 3 - Risk/UQ Training Knobs
Keep all experiment knobs centralized here.


In [ ]:
# Risk/UQ experiment knobs
cfg.risk_dataset_build = True
cfg.risk_dataset_candidate_count = 8
cfg.risk_dataset_control_horizon_steps = 6
cfg.risk_dataset_label_horizons = (5, 10, 15)
cfg.risk_dataset_events = ('collision', 'offroad', 'failure_proxy')

cfg.risk_model_ensemble_size = 5
cfg.risk_model_hidden_dims = (128, 128)
cfg.risk_model_dropout = 0.10
cfg.risk_model_learning_rate = 1e-3
cfg.risk_model_batch_size = 1024
cfg.risk_model_max_epochs = 50
cfg.risk_model_patience = 8
cfg.risk_model_checkpoint_every_epochs = 1
cfg.risk_model_resume_from_checkpoints = True

cfg.risk_calibration_method = 'temperature'
cfg.risk_conformal_alpha = 0.10
cfg.risk_control_fail_budget = 0.20

cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg.uq_eval_probability_bins = 15
cfg


## Step 4 - Mandatory Quick Probe Gate
Do not proceed if the surprise signal collapses.


In [ ]:
from src.workflows import report_quick_probe_bundle, run_quick_probe_with_auto_escalation

quick_probe_settings = {
    'quick_probe_scenarios': 1,
    'quick_probe_proposals_per_scenario': 4,
    'stop_if_quick_probe_collapsed': True,
    'auto_escalate_quick_probe': True,
    'max_probe_escalations': 2,
    'probe_scale_multipliers': (1.0, 1.35),
    'probe_delta_l2_multipliers': (1.0, 1.2),
    'probe_delta_clip_multipliers': (1.0, 1.1),
    'probe_persist_step_multipliers': (1.0, 2.0),
    'probe_interaction_gain_multipliers': (1.0, 1.5),
    'probe_budget_bump_per_escalation': 2,
    'probe_force_behavioral_preset': True,
    'probe_target_top_k': 3,
    'probe_hist_prim_selector_mode': 'interaction_band',
    'probe_surprise_metrics': ('predictive_seq_w2', 'predictive_seq_kl', 'predictive_w2'),
    'probe_belief_source_modes': ('auto', 'b1', 'b2', 'b3'),
    'probe_metric_selection_policy': 'best_feasible_score',
    'probe_min_nonzero_surprise_fraction': 0.01,
    'probe_min_realized_fraction': 0.10,
    'probe_min_effect_l2_mean': 0.05,
    'probe_min_belief_shift_mean': 1e-8,
    'probe_min_policy_shift_mean': 1e-8,
    'probe_min_realization_ratio_mean': 0.05,
    'probe_min_raw_belief_shift_mean': 1e-7,
    'probe_min_raw_policy_shift_mean': 1e-7,
    'probe_min_raw_fraction_of_total': 0.20,
    'probe_require_belief_and_policy': True,
    'probe_require_raw_signal': True,
    'apply_successful_probe_tuning': True,
}

probe_bundle = run_quick_probe_with_auto_escalation(
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    run_quick_surprise_probe_enabled=True,
    build_simulation_context=True,
    **quick_probe_settings,
)

cfg = probe_bundle.cfg
search_cfg = probe_bundle.search_cfg
report_quick_probe_bundle(probe_bundle, search_cfg=search_cfg, display_fn=display)

if bool(probe_bundle.final_collapsed):
    raise RuntimeError(f"Quick probe collapsed: {probe_bundle.signal_failure_reasons}")

runner = probe_bundle.runner
eval_idx = probe_bundle.eval_idx
reference_df = probe_bundle.reference_df
if runner is None:
    raise RuntimeError('Quick probe did not return simulation context. Cannot continue.')


## Step 5 - Mandatory Preflight Gate
Preflight must pass before running dataset build/training.


In [ ]:
from src.workflows import report_preflight_bundle, run_preflight_bundle

RESTORE_FROM_UPLOAD = False
STOP_ON_PREFLIGHT_FAIL = True

preflight_bundle = run_preflight_bundle(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=bool(RESTORE_FROM_UPLOAD),
    stop_on_preflight_fail=bool(STOP_ON_PREFLIGHT_FAIL),
)
report_preflight_bundle(preflight_bundle, display_fn=display)

if not bool(preflight_bundle.ready_for_main_loop):
    raise RuntimeError('Preflight failed; fix checks before risk model training.')


## Step 6 - Write Contract Manifest (Gates Passed)
This records run metadata required for reproducibility and downstream notebook checks.


In [ ]:
from src.workflows import write_notebook_contract_manifest

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='risk_model_training_colab',
    stage='gates_passed',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(not probe_bundle.final_collapsed),
    preflight_pass=bool(preflight_bundle.ready_for_main_loop),
)
print('manifest_path =', manifest_path)


## Step 7 - Run Resume-Aware Risk Training Flow
If artifacts already exist in `auto/resume`, they are reused; otherwise training resumes from checkpoints.


In [ ]:
import pandas as pd
from src.workflows import (
    has_existing_risk_model_artifacts,
    has_existing_risk_training_checkpoints,
    load_existing_risk_dataset_artifact,
    run_risk_training_flow,
)

existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
existing_model_ready = has_existing_risk_model_artifacts(cfg.run_prefix)
existing_checkpoint_ready = has_existing_risk_training_checkpoints(cfg.run_prefix)

if RUN_MODE == 'fresh':
    existing_dataset_df = pd.DataFrame()

print('existing_model_ready =', existing_model_ready)
print('existing_checkpoint_ready =', existing_checkpoint_ready)

if existing_dataset_df.empty:
    print('[risk-train] no persisted dataset found; building from current runner context...')
    bundle = run_risk_training_flow(
        cfg=cfg,
        runner=runner,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )
else:
    print(f'[risk-train] using persisted dataset rows={len(existing_dataset_df)}')
    bundle = run_risk_training_flow(
        cfg=cfg,
        dataset_df=existing_dataset_df,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )

print('loaded_from_existing =', bundle.loaded_from_existing)
print('dataset_rows =', len(bundle.dataset_bundle.dataset_df))
print('artifact_keys =', sorted(bundle.artifact_paths.keys()))


## Step 8 - Evaluation and Diagnostics
Use these outputs as the run report section for this notebook.


In [ ]:
display(bundle.training_bundle.train_summary.head(20))
if bundle.calibration_bundle is not None:
    display(bundle.calibration_bundle.summary_df)
    display(bundle.calibration_bundle.reliability_df.head(20))


## Step 9 - Update Manifest (Training Completed)
This marks the notebook run complete for downstream benchmark/export notebooks.


In [ ]:
manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='risk_model_training_colab',
    stage='risk_training_completed',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(not probe_bundle.final_collapsed),
    preflight_pass=bool(preflight_bundle.ready_for_main_loop),
    extra_fields={
        'loaded_from_existing': int(bool(bundle.loaded_from_existing)),
        'artifact_keys': sorted(list(bundle.artifact_paths.keys())),
    },
)
print('manifest_path =', manifest_path)
